# Chapter 1: Regression — Predicting Continuous Outcomes

### 1.1 Simple Linear Regression

In [ ]:
np.random.seed(42)
ad_spend = np.linspace(50, 300, 40)                            # ₹ '000
revenue = 2.8 * ad_spend + 40 + np.random.normal(0, 25, 40)    # ₹ '000

df = pd.DataFrame({"Ad_Spend": ad_spend, "Revenue": revenue})

X = df[["Ad_Spend"]]      # scikit-learn ALWAYS wants X as 2D (a DataFrame or 2D array), even for one feature
y = df["Revenue"]          # y can be 1D (a Series or array)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.2f}")
print(f"Coefficient (lift per ₹1k ad spend): {model.coef_[0]:.2f}")

y_pred = model.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

⚠️ `X = df[["Ad_Spend"]]` uses **double** brackets deliberately — scikit-learn requires the feature matrix `X` to be 2-dimensional (rows × columns) even when you only have one feature. `df["Ad_Spend"]` (single brackets) returns a 1D Series and will raise an error when passed to `.fit()`. This is one of the single most common first-time errors when starting with scikit-learn.

**`random_state=42`** appears in nearly every scikit-learn function that involves any randomness (splitting data, initializing clusters, etc.). Setting it to a fixed number makes your results **reproducible** — every time you (or a colleague) rerun the exact same code, you get the exact same split/result, which matters enormously when debugging or comparing model variants fairly. The number 42 itself carries no special meaning beyond being a widely-used convention (a nod to "The Hitchhiker's Guide to the Galaxy") — any fixed integer works identically.

**RMSE (Root Mean Squared Error)** is in the same units as your target variable (₹, here) and represents roughly "how far off is a typical prediction" — more directly interpretable to a business audience than R² alone. Report both: R² for "how much of the variation does this model explain overall," RMSE for "what's the typical dollar-value error I should expect."

### 1.2 Multiple Linear Regression

In [ ]:
np.random.seed(42)
n = 100
data = pd.DataFrame({
    "Ad_Spend": np.random.uniform(50, 300, n),
    "Seasonality": np.random.randint(1, 5, n),        # 1=Q1 ... 4=Q4
    "Comp_Price": np.random.uniform(80, 150, n)
})
data["Revenue"] = (2.5*data["Ad_Spend"] + 15*data["Seasonality"]
                    - 1.2*data["Comp_Price"] + 50 + np.random.normal(0, 20, n))

X = data[["Ad_Spend", "Seasonality", "Comp_Price"]]
y = data["Revenue"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

multi = LinearRegression()
multi.fit(X_train, y_train)
for feat, coef in zip(X.columns, multi.coef_):
    print(f"  {feat}: {coef:.2f}")
print(f"R² on test: {r2_score(y_test, multi.predict(X_test)):.3f}")

Adding predictors is nearly free syntactically — just pass more columns in `X`. The interpretation deepens exactly as it did with Statsmodels in Volume 04: each coefficient represents that feature's effect *holding the others constant*. Note the negative coefficient on `Comp_Price` — the model correctly recovers that higher competitor pricing is associated with *higher* revenue for us (a sensible business relationship: when competitors raise prices, customers shift toward us), which is a good example of a model finding an interpretable, sensible pattern rather than a black box.

### 1.3 Polynomial Regression — Curved Relationships

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

x = np.linspace(0, 10, 40).reshape(-1, 1)     # reshape needed: turn a 1D array into a 2D column
y_curved = 0.5*x.ravel()**2 - 3*x.ravel() + 8 + np.random.normal(0, 3, 40)

poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_model.fit(x, y_curved)

plt.scatter(x, y_curved, label="Actual")
plt.plot(x, poly_model.predict(x), color="red", label="Polynomial fit")
plt.legend()
plt.title("Polynomial Regression Example")
plt.show()
print("R²:", r2_score(y_curved, poly_model.predict(x)))

Not every business relationship is a straight line. **Advertising saturation** (returns diminish as spend increases), **learning curves**, and many pricing effects are curved. `PolynomialFeatures(degree=2)` doesn't require a different model at all — it simply *engineers new columns* (x², in addition to x) and then fits an ordinary `LinearRegression` on top of the expanded feature set. This is a genuinely important insight about "linear" regression: it's linear in its *coefficients*, not necessarily in the original variable — which is exactly why you can fit curves with it once you feed it the right engineered features. `make_pipeline` chains the feature transformation and the model into a single object so you call `.fit()`/`.predict()` once instead of two separate steps — a small preview of the full `Pipeline` machinery covered in Chapter 5.

⚠️ Higher polynomial degrees fit the training data increasingly well but risk **overfitting** — capturing noise rather than the true underlying pattern, and performing worse on genuinely new data. Degree 2 or 3 covers the vast majority of real curved business relationships; reach for anything higher only with strong justification and careful validation.

### 1.4 Why Train/Test Split Exists — and Why It's Non-Negotiable

In [ ]:
# The single most important rule in this entire volume:
# NEVER evaluate a model on the same data it was trained on.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

If you evaluate a model on the same data you trained it on, you're measuring how well it **memorized** that data, not how well it will perform on genuinely new data it hasn't seen — a distinction called **overfitting**, and it is the single most common way novice ML projects produce misleadingly optimistic results that fall apart in production. `test_size=0.2` or `0.25` (holding out 20–25% of your data, untouched, until the very end) is the standard convention; `stratify=y` (seen in Chapter 2 below) additionally ensures the train and test sets have the same proportion of each class, which matters when one outcome (e.g., churn) is rarer than the other.

### Cheat Sheet — Regression

| Task | Code |
|---|---|
| Feature matrix (always 2D!) | `X = df[["col"]]` or `df[["col1","col2"]]` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| Fit | `model = LinearRegression(); model.fit(X_train, y_train)` |
| Predict | `model.predict(X_test)` |
| Coefficients | `model.coef_`, `model.intercept_` |
| Evaluate | `r2_score(y_test, y_pred)`, `np.sqrt(mean_squared_error(y_test, y_pred))` |
| Curved relationships | `make_pipeline(PolynomialFeatures(degree=2), LinearRegression())` |

---